# Preqin Auth Token Generation

Authenticates with the Preqin OAuth2 endpoint and stores the bearer token as a task value for downstream tasks.

| Task Value | Description |
|---|---|
| `access_token` | Bearer token passed to all API call tasks via For Each sub-job |


In [0]:
import requests
import time

# ---------------------------------------------------------------------------
# Authenticate with Preqin OAuth2 endpoint
# Retries on transient errors (5xx, 429, network) only.
# Fails immediately on 4xx — bad credentials are not retryable.
# ---------------------------------------------------------------------------

TOKEN_URL   = "https://id.preqin.com/connect/token"
scope = "preqin.feeds.api.v1.default"
MAX_RETRIES = 3

payload = {
    "grant_type":    "password",
    "client_id":     dbutils.secrets.get("preqin-client-id", "client_id"),
    "client_secret": dbutils.secrets.get("preqin-client-secret", "client_secret"),
    "username":      dbutils.secrets.get("preqin-username", "username"),
    "password":      dbutils.secrets.get("preqin-pw", "password"),
    "scope":         "preqin.feeds.api.v1.default",
}

for attempt in range(1, MAX_RETRIES + 1):
    try:
        response = requests.post(TOKEN_URL, data=payload, timeout=30)

        # 4xx — non-retryable: bad credentials, forbidden, etc.
        if 400 <= response.status_code < 500:
            raise RuntimeError(
                f"Authentication failed ({response.status_code}) — "
                f"check credentials in secret scope 'preqin'"
            )

        response.raise_for_status()  # raises on 5xx

        token = response.json().get("access_token")
        if not token:
            raise ValueError(f"access_token missing in response: {response.text[:200]}")

        dbutils.jobs.taskValues.set(key="access_token", value=token)
        print(f"Token generated successfully (attempt {attempt})")
        break

    except RuntimeError:
        raise  # non-retryable — propagate immediately

    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"Attempt {attempt}/{MAX_RETRIES} failed: {e}")
        if attempt < MAX_RETRIES:
            time.sleep(2 ** attempt)
        else:
            raise RuntimeError(f"Failed to generate token after {MAX_RETRIES} attempts") from e

Token generated successfully (attempt 1)


In [0]:
dbutils.jobs.taskValues.set(key="access_token", value=token)